# CS781 Phase 1 — Prepare Test Set

**Set accelerator to None** -- nothing here needs a GPU, so this costs zero GPU-hours and can run in parallel with a training session.

Test set is small (hundreds of short prompts), so tokenizing it isn't a speed problem the way the 4.5B-token training corpus was -- the real value here is different: the dataset repo says the test file *"is a dynamic document and will keep getting updated"*. Downloading a snapshot now, before generation actually runs, means whatever you submit is generated against a fixed, known version of the test set, not whatever happens to be live at generation time.

**Input to attach**: the tokenizer dataset.

## 1. Hugging Face Login

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Hugging Face login successful.")

## 2. Configuration

In [ ]:
import glob, json, shutil, time

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

OUTPUT_DIR = "/kaggle/working/cs781_phase1_test_snapshot"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Download and Snapshot the Test File

In [ ]:
from huggingface_hub import hf_hub_download

downloaded_path = hf_hub_download(
    repo_id=HF_DATASET_REPO, filename=TEST_FILE, repo_type="dataset", token=HF_TOKEN
)
print("Downloaded:", downloaded_path)

SNAPSHOT_PATH = os.path.join(OUTPUT_DIR, "test_snapshot.json")
shutil.copy(downloaded_path, SNAPSHOT_PATH)

with open(SNAPSHOT_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)
print(f"Snapshotted {len(test_data)} test examples to {SNAPSHOT_PATH}")
print(f"Snapshot taken: {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())}")

## 4. Real Prompt Statistics

Uses the actual tokenizer, so these are real numbers for planning generation time and context length -- not estimates.

In [ ]:
from transformers import AutoTokenizer

def find_dirs(filename):
    return sorted({os.path.dirname(p) for p in glob.glob(f"/kaggle/input/**/{filename}", recursive=True)})

cands = [d for d in find_dirs("tokenizer.json") if "checkpoint-" not in d and "legal-gqa-final" not in d]
if not cands:
    raise FileNotFoundError("No tokenizer dataset found -- attach it via Add Input.")
TOKENIZER_DIR = ([d for d in cands if os.path.exists(os.path.join(d, "trained_on.json"))] or cands)[0]
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR)
print("Tokenizer:", TOKENIZER_DIR, "| vocab size:", len(tokenizer))

word_counts, token_counts = [], []
for rec in test_data.values():
    text = rec["text"]
    word_counts.append(len(text.split()))
    token_counts.append(len(tokenizer.encode(text, add_special_tokens=False)))

def stats(xs, label):
    xs = sorted(xs)
    n = len(xs)
    print(f"{label}: min={xs[0]} max={xs[-1]} mean={sum(xs)/n:.1f} median={xs[n//2]} "
          f"p90={xs[int(n*0.9)]} p99={xs[min(n-1, int(n*0.99))]}")

print(f"\n{len(test_data)} test prompts")
stats(word_counts, "words/prompt")
stats(token_counts, "tokens/prompt")

max_tokens = max(token_counts)
print(f"\nmax tokens/prompt + 128-word generation budget (~190 tokens): ~{max_tokens + 190} "
      f"-- confirm this fits under model 1's context length (1024)")

with open(os.path.join(OUTPUT_DIR, "test_stats.json"), "w") as f:
    json.dump({
        "num_examples": len(test_data), "snapshot_time_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "word_counts": {"min": min(word_counts), "max": max(word_counts), "mean": sum(word_counts)/len(word_counts)},
        "token_counts": {"min": min(token_counts), "max": max(token_counts), "mean": sum(token_counts)/len(token_counts)},
    }, f, indent=2)
print("\nSaved stats to", os.path.join(OUTPUT_DIR, "test_stats.json"))